# Week 37: Structure Span Prediction

In [ ]:
import nb_setup
import numpy as np
import pandas as pd
from src.features import featurise_df
from src.utils import split_by_lengths
from src.data import load_tokenised_data
from src.evaluation import evaluate_sequences
from sklearn.linear_model import LogisticRegression
from sklearn.feature_extraction import DictVectorizer
from src.bio_conversion import create_bio_labels, test_bio_conversion

In [ ]:
df_train_tokenised, df_val_tokenised = load_tokenised_data()

### BIO Labelling and Validation

In [ ]:
bio_labels_train = df_train_tokenised.apply(
    lambda row: create_bio_labels(
        row["answerable"],
        row["context"],
        row["context_tokens"],
        row["context_offsets"],
        row["answer_start"],
        row["answer"],
    ),
    axis=1,
)
df_train_tokenised["bio_labels"], df_train_tokenised["bio_alignment"] = zip(*bio_labels_train)

bio_labels_val = df_val_tokenised.apply(
    lambda row: create_bio_labels(
        row["answerable"],
        row["context"],
        row["context_tokens"],
        row["context_offsets"],
        row["answer_start"],
        row["answer"],
    ),
    axis=1,
)
df_val_tokenised["bio_labels"], df_val_tokenised["bio_alignment"] = zip(*bio_labels_val)

In [ ]:
df_train_tokenised["bio_alignment"].value_counts()

In [ ]:
test_bio_conversion()

### Question-conditioned Sequence Labeller

In [ ]:
X_train_dicts, y_train, train_lengths = featurise_df(
    df_train_tokenised
)

X_val_dicts, y_val, val_lengths = featurise_df(
    df_val_tokenised
)

In [ ]:
vectorizer = DictVectorizer()

X_train = vectorizer.fit_transform(X_train_dicts)
X_val = vectorizer.transform(X_val_dicts)

In [ ]:
classifier = LogisticRegression(
    max_iter=1000,
    class_weight={
        0: 1,
        1: 3,
        2: 3,
    },
)

classifier.fit(
    X_train,
    y_train,
)

In [ ]:
y_pred = classifier.predict(X_val)

pred_sequences = split_by_lengths(
    y_pred,
    val_lengths,
)

gold_sequences = split_by_lengths(
    y_val,
    val_lengths,
)

languages = df_val_tokenised["lang"].tolist()

lr_results = evaluate_sequences(
    pred_sequences,
    gold_sequences,
    languages,
    model_name="Feature-based LR",
)

In [ ]:
empty_predictions = [
    np.zeros(len(gold), dtype=int)
    for gold in gold_sequences
]

baseline_results = evaluate_sequences(
    empty_predictions,
    gold_sequences,
    languages,
    model_name="Empty Baseline",
)

In [ ]:
results_df = pd.concat(
    [
        baseline_results,
        lr_results,
    ],
    ignore_index=True,
)
results_df[
    ["Precision", "Recall", "Token F1", "Exact Span Match"]
] = results_df[
    ["Precision", "Recall", "Token F1", "Exact Span Match"]
].round(4)

results_df